# Pipeline de Meta-Learning para IQA

Este notebook executa o fluxo principal do projeto de Meta-Learning aplicado à Avaliação de Qualidade de Imagem (IQA). O pipeline está dividido em etapas lógicas, desde a leitura dos dados até a geração dos gráficos de avaliação.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from scipy.stats import rankdata

import sys
sys.path.append('../src')

from data_loader import build_full_dataframe
from evaluation import get_metrics_dict, calculate_performance_matrix
from features import build_meta_features_matrix
from models import BaselineRanker, SignificantWinsRanker, MetaRegressor, HarrisForest
from experiments import leave_one_dataset_out_evaluation, plot_critical_difference_diagram

base_dir = Path.cwd().parent
data_raw_dir = base_dir / 'data' / 'raw'
data_processed_dir = base_dir / 'data' / 'processed'
notebooks_dir = base_dir / 'notebooks'

data_processed_dir.mkdir(parents=True, exist_ok=True)
notebooks_dir.mkdir(parents=True, exist_ok=True)

### 1. Carregamento de Metadados e Matriz de Performance (P)

Lemos os dados brutos e calculamos a performance de cada métrica de IQA para os conjuntos de dados. O resultado é consolidado através da média entre os folds.

In [ ]:
df_metadata = build_full_dataframe(data_raw_dir)

metrics = get_metrics_dict()
df_P_folds = calculate_performance_matrix(df_metadata, metrics)
df_P_folds.to_csv(data_processed_dir / 'P_matrix_folds.csv', index=False)

P_matrix = df_P_folds.groupby('dataset')[list(metrics.keys())].mean()

### 2. Extração de Meta-características (X) e Ranks (R)

Aqui calculamos as meta-características (X) que descrevem cada dataset. Também convertemos as performances em matriz de ranks (R), onde o algoritmo com melhor correlação recebe rank 1.

In [ ]:
df_X = build_meta_features_matrix(df_metadata)
df_X = df_X.sort_values('dataset').reset_index(drop=True)
df_X.to_csv(data_processed_dir / 'X_matrix.csv', index=False)
X_matrix = df_X.drop('dataset', axis=1)

R_matrix = P_matrix.apply(lambda row: rankdata(-row, method='average'), axis=1, result_type='broadcast')
R_matrix.to_csv(data_processed_dir / 'R_matrix.csv', index=True)

### 3. Avaliação e Gráficos

Inicializamos os modelos de Meta-Learning (baselines e propostas como o HARRIS) e avaliamos utilizando validação "Leave-One-Dataset-Out". Os resultados são apresentados em gráficos de curva de perda e diagrama de diferença crítica (Friedman + Nemenyi).

In [ ]:
models = {
    'AR': BaselineRanker(method='AR'),
    'MR': BaselineRanker(method='MR'),
    'Abordagem1 (P)': MetaRegressor(target='P'),
    'Abordagem2 (R)': MetaRegressor(target='R'),
    'HARRIS (L=0.0)': HarrisForest(lambd=0.0),
    'HARRIS (L=0.5)': HarrisForest(lambd=0.5),
    'HARRIS (L=1.0)': HarrisForest(lambd=1.0),
}

results = leave_one_dataset_out_evaluation(X_matrix, P_matrix, R_matrix, models)

plt.figure(figsize=(10, 6))
for name, res in results.items():
    plt.plot(range(1, len(res['Mean_Curve'])+1), res['Mean_Curve'], label=f"{name} (AUC={res['Mean_AUC']:.3f})")
plt.xlabel('Número de testes (t)')
plt.ylabel('Perda Média')
plt.legend()
plt.title('Curvas de Perda das Abordagens')
plt.savefig(notebooks_dir / 'loss_curves.png')
plt.show()
plt.close()

dict_srcc = {name: res['All_SRCC'] for name, res in results.items()}
plot_critical_difference_diagram(dict_srcc, alpha=0.05, save_path=notebooks_dir / 'cd_diagram.png')